In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display, Latex
from scipy.stats import f, norm


Question 1

In [17]:
df = pd.read_csv("ps1small.csv")
df["lwage"] = np.log(df["wage"])

In [18]:
df["age"].agg(["min", "max"])

min    26
max    30
Name: age, dtype: int64

In [19]:
df["education"].agg(["min", "max"])

min    12
max    17
Name: education, dtype: int64

In [20]:
for i in range(12, 18):
    for j in range(26, 31):
        df[f"d{i}{j}"] = ((df["education"] == i) & (df["age"] == j)).astype(int)


In [21]:
d_cols = [col for col in df.columns if col.startswith("d")]
D = df[d_cols]

model = sm.OLS(df["lwage"], D).fit()

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.089
Model:                            OLS   Adj. R-squared:                  0.059
Method:                 Least Squares   F-statistic:                     2.938
Date:                Mon, 26 Oct 2026   Prob (F-statistic):           4.90e-07
Time:                        16:59:57   Log-Likelihood:                -1099.1
No. Observations:                 898   AIC:                             2258.
Df Residuals:                     868   BIC:                             2402.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
d1226          9.4513      0.091    104.112      0.0

In [22]:
print(pd.DataFrame({
    "estimate": model.params,
    "standard_error": model.bse
}))

        estimate  standard_error
d1226   9.451320        0.090780
d1227   9.300426        0.084980
d1228   9.537856        0.083280
d1229   9.385826        0.085421
d1230   9.628323        0.087737
d1326   9.612120        0.192010
d1327   9.755901        0.192010
d1328   9.378158        0.174517
d1329   9.819982        0.223685
d1330   9.446488        0.202991
d1426   9.580658        0.223685
d1427   9.794423        0.170843
d1428   9.642237        0.216101
d1429   9.712199        0.174517
d1430   9.645440        0.170843
d1526   9.534555        0.241608
d1527   9.385619        0.232129
d1528   9.573155        0.264668
d1529   9.540361        0.252351
d1530   9.919473        0.241608
d1626  10.070500        0.167391
d1627   9.944426        0.187149
d1628   9.991987        0.147954
d1629  10.178524        0.130710
d1630  10.070591        0.164140
d1726   9.958317        0.483216
d1727  10.045760        0.223685
d1728  10.165648        0.295908
d1729   9.617564        0.418477
d1730  10.

In [23]:

table = pd.DataFrame({
    "term": model.params.index,
    "Estimate": model.params.to_numpy(),
    "Standard error": model.bse.to_numpy()
})

# Turn labels such as d1226 into education=12, age=26
table[["Education", "Age"]] = (
    table["term"].str.extract(r"^d(\d{2})(\d{2})$").astype(int)
)

table = table[["Education", "Age", "Estimate", "Standard error"]]

latex_table = table.to_latex(
    index=False,
    column_format="rrrr",
    float_format=lambda x: f"{x:.3f}",
    caption="OLS estimates of mean log wage by education and age",
    label="tab:mean_log_wage",
    position="htbp"
)

print(latex_table)

\begin{table}[htbp]
\caption{OLS estimates of mean log wage by education and age}
\label{tab:mean_log_wage}
\begin{tabular}{rrrr}
\toprule
Education & Age & Estimate & Standard error \\
\midrule
12 & 26 & 9.451 & 0.091 \\
12 & 27 & 9.300 & 0.085 \\
12 & 28 & 9.538 & 0.083 \\
12 & 29 & 9.386 & 0.085 \\
12 & 30 & 9.628 & 0.088 \\
13 & 26 & 9.612 & 0.192 \\
13 & 27 & 9.756 & 0.192 \\
13 & 28 & 9.378 & 0.175 \\
13 & 29 & 9.820 & 0.224 \\
13 & 30 & 9.446 & 0.203 \\
14 & 26 & 9.581 & 0.224 \\
14 & 27 & 9.794 & 0.171 \\
14 & 28 & 9.642 & 0.216 \\
14 & 29 & 9.712 & 0.175 \\
14 & 30 & 9.645 & 0.171 \\
15 & 26 & 9.535 & 0.242 \\
15 & 27 & 9.386 & 0.232 \\
15 & 28 & 9.573 & 0.265 \\
15 & 29 & 9.540 & 0.252 \\
15 & 30 & 9.919 & 0.242 \\
16 & 26 & 10.070 & 0.167 \\
16 & 27 & 9.944 & 0.187 \\
16 & 28 & 9.992 & 0.148 \\
16 & 29 & 10.179 & 0.131 \\
16 & 30 & 10.071 & 0.164 \\
17 & 26 & 9.958 & 0.483 \\
17 & 27 & 10.046 & 0.224 \\
17 & 28 & 10.166 & 0.296 \\
17 & 29 & 9.618 & 0.418 \\
17 & 30 & 10.016 

In [24]:
X = df[["education", "age"]].copy()
X["education_age"] = X["education"] * X["age"]
X = sm.add_constant(X)

restricted_model = sm.OLS(df["lwage"], X).fit()
print(restricted_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.064
Model:                            OLS   Adj. R-squared:                  0.061
Method:                 Least Squares   F-statistic:                     20.48
Date:                Mon, 26 Oct 2026   Prob (F-statistic):           7.69e-13
Time:                        16:59:57   Log-Likelihood:                -1111.3
No. Observations:                 898   AIC:                             2231.
Df Residuals:                     894   BIC:                             2250.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             6.1075      4.647      1.314

In [25]:
F_stat, p_value, df_diff = model.compare_f_test(restricted_model)

print(f"F({int(df_diff)}, {int(model.df_resid)}) = {F_stat:.4f}")
print(f"p-value = {p_value:.4f}")

F(26, 868) = 0.9199
p-value = 0.5806


Question 3 

In [26]:
wage_data = pd.read_excel("wage.xlsx")
wage_data["log_wage"] = np.log(wage_data["wage"])


In [27]:
# Group 1 = females; group 2 = males
female = wage_data.loc[wage_data["male"] == 0, "log_wage"].to_numpy()
male = wage_data.loc[wage_data["male"] == 1, "log_wage"].to_numpy()

In [28]:

n_female = len(female)
n_male = len(male)
n_total = n_female + n_male

female_variance = female.var(ddof=1)
male_variance = male.var(ddof=1)


In [29]:

# Test statistic T
T_stat = np.sqrt(n_female * n_male / n_total) * (
    np.log(female_variance) - np.log(male_variance)
)

# Normal-theory F-test
variance_ratio = female_variance / male_variance
dof_female = n_female - 1
dof_male = n_male - 1

F_critical = f.ppf(0.95, dof_female, dof_male)
p_value_F = f.sf(variance_ratio, dof_female, dof_male)

# Kurtosis estimate and robust test
female_residuals = female - female.mean()
male_residuals = male - male.mean()

kurtosis_hat = n_total * (
    np.sum(female_residuals**4) + np.sum(male_residuals**4)
) / (
    np.sum(female_residuals**2) + np.sum(male_residuals**2)
)**2

z_robust = T_stat / np.sqrt(kurtosis_hat - 1)
p_value_robust = norm.sf(z_robust)


In [30]:
print(f"Female log-wage variance: {female_variance:.5f}")
print(f"Male log-wage variance: {male_variance:.5f}")


print(f"F critical value: {F_critical:.5f}")
print(f"Normal-theory F-test p-value: {p_value_F:.5f}")
print(f"T statistic: {T_stat:.5f}")

print(f"Estimated kurtosis: {kurtosis_hat:.5f}")
print(f"Robust z statistic: {z_robust:.5f}")
print(f"Robust test p-value: {p_value_robust:.5f}")

Female log-wage variance: 0.39788
Male log-wage variance: 0.36641
F critical value: 1.08446
Normal-theory F-test p-value: 0.04735
T statistic: 2.36210
Estimated kurtosis: 7.37812
Robust z statistic: 0.93530
Robust test p-value: 0.17482
